In [8]:
import pandas as pd
import numpy as np
import os

def limpiar_monto(valor):
    if pd.isna(valor) or valor == "" or str(valor).strip() == "0,00 €":
        return 0.0
    # Eliminar símbolo de euro y espacios
    s = str(valor).replace('€', '').strip()
    # Cambiar formato europeo (1.234,56) a estándar (1234.56)
    s = s.replace('.', '').replace(',', '.')
    try:
        return float(s)
    except:
        return 0.0

# Diccionario para convertir nombres de meses a números
meses_map = {
    "Enero": "01", "Febrero": "02", "Marzo": "03", "Abril": "04",
    "Mayo": "05", "Junio": "06", "Julio": "07", "Agosto": "08",
    "Septiembre": "09", "Octubre": "10", "Noviembre": "11", "Diciembre": "12"
}

In [29]:
import pandas as pd
import numpy as np
import csv

def procesar_csv_gastos(file_path, year, month):
    df_raw = pd.read_csv(file_path, header=None)
    
    # 1. Identificar filas de Ingresos y Gastos
    idx_ingresos = df_raw[df_raw[0] == 'Ingresos'].index[0]
    idx_gastos = df_raw[df_raw[0] == 'Gastos'].index[0]
    
    data_list = []

    def extraer_movimientos(start_idx, end_idx, tipo_movimiento):
        for i in range(start_idx, end_idx):
            categoria_orig = str(df_raw.iloc[i, 1]).strip() 
            if pd.isna(categoria_orig) or categoria_orig == "" or categoria_orig == "nan": continue
            
            for dia_col in range(2, 33):
                valor = df_raw.iloc[i, dia_col]
                
                if pd.notna(valor) and valor != "0,00 €" and valor != 0:
                    # Limpieza del importe
                    importe_str = str(valor).replace(' €', '').replace('.', '').replace(',', '.')
                    try:
                        importe = float(importe_str)
                        if importe == 0: continue
                    except: continue
                    
                    dia = dia_col - 1 
                    fecha = f"{dia:02d}/{month:02d}/{year}"
                    
                    # --- LÓGICA DE CUSTOMIZACIÓN ---
                    desc_final = categoria_orig
                    cat_final = categoria_orig
                    
                    if tipo_movimiento == "Ingreso":
                        cat_final = "Billetazo"
                        if categoria_orig == "Banca":
                            desc_final = "Bizum"
                    else:
                        # Regla especial para el Café (1.20 - 1.30 en Comida)
                        if categoria_orig == "Comida" and 1.20 <= importe <= 1.30:
                            cat_final = "Ocio"
                            desc_final = "Café"
                        # Regla para Banca -> Otro
                        elif categoria_orig == "Banca":
                            cat_final = "Otro"
                        # Regla para Entretenimiento -> Ocio
                        elif categoria_orig == "Entretenimiento":
                            cat_final = "Ocio"
                    
                    data_list.append({
                        "Fecha": fecha,
                        "Tipo": tipo_movimiento,
                        "Categoría": cat_final,
                        "Descripción": desc_final,
                        "Importe": f"{importe:.2f}".replace('.', ','),
                        "Compromiso": "Variable",
                        "Estado": "Pagado"
                    })

    extraer_movimientos(idx_ingresos, idx_gastos, "Ingreso")
    extraer_movimientos(idx_gastos, len(df_raw), "Gasto")

    return pd.DataFrame(data_list)

# --- EJECUCIÓN ---
df_resultado = procesar_csv_gastos('../data/2022/2022 - Diciembre.csv', 2022, 12)

df_resultado.to_csv('mes_limpio.csv', 
                    index=False, 
                    sep=',', 
                    quoting=csv.QUOTE_NONNUMERIC,
                    encoding='utf-8-sig')

print("¡Procesado! Los cafés de 1,20€-1,30€ ahora son Ocio.")

¡Procesado! Los cafés de 1,20€-1,30€ ahora son Ocio.


In [ ]:
import pandas as pd
import numpy as np
import csv
import os

def procesar_csv_2023(file_path, year, month):
    # Leer el CSV
    df_raw = pd.read_csv(file_path, header=None)
    
    # 1. LOCALIZAR LA FILA DE DÍAS (dinámico para 2023)
    # Buscamos la fila que tenga un 0 o '0' en las primeras columnas
    try:
        idx_dias = df_raw[(df_raw[2] == '0') | (df_raw[2] == 0)].index[0]
    except:
        # Si falla, buscamos donde el valor sea 1 (segunda columna de datos)
        idx_dias = df_raw[(df_raw[3] == '1') | (df_raw[3] == 1)].index[0]

    # 2. LOCALIZAR SECCIONES
    idx_ingresos = df_raw[df_raw[0] == 'Ingresos'].index[0]
    idx_gastos = df_raw[df_raw[0] == 'Gastos'].index[0]
    
    data_list = []

    def extraer_movimientos(start_idx, end_idx, tipo_movimiento):
        for i in range(start_idx, end_idx):
            # En 2023 la categoría suele estar en la columna 1 (B)
            categoria_orig = str(df_raw.iloc[i, 1]).strip() 
            if categoria_orig.lower() in ["nan", "", "total", "ahorro"]: continue
            
            # Recorrer columnas de días (de la 2 hasta la penúltima)
            for col in range(2, df_raw.shape[1] - 1):
                valor = df_raw.iloc[i, col]
                
                if pd.notna(valor) and str(valor).strip() not in ["0,00 €", "0", "0.0", ""]:
                    try:
                        # Limpieza de importe
                        val_str = str(valor).replace(' €', '').replace('.', '').replace(',', '.')
                        importe = abs(float(val_str))
                        if importe == 0: continue
                    except: continue
                    
                    # Obtener el día real de la fila idx_dias
                    try:
                        dia = int(float(str(df_raw.iloc[idx_dias, col]).strip()))
                    except: continue
                    
                    fecha = f"{dia:02d}/{month:02d}/{year}"
                    
                    # --- REGLAS DE PERSONALIZACIÓN ---
                    desc_final = categoria_orig
                    cat_final = categoria_orig
                    
                    if tipo_movimiento == "Ingreso":
                        cat_final = "Billetazo"
                        if categoria_orig == "Banca" or categoria_orig == "Ingreso":
                            # Si es el nombre genérico, lo dejamos como la descripción que traiga o Bizum
                            desc_final = "Bizum" if categoria_orig == "Banca" else "Ingreso"
                    else:
                        # Regla del Café (1,20€ - 1,30€ en Comida)
                        if categoria_orig == "Comida" and 1.20 <= importe <= 1.35:
                            cat_final = "Ocio"
                            desc_final = "Café"
                        # Regla Banca -> Otro
                        elif categoria_orig == "Banca":
                            cat_final = "Otro"
                        # Regla Entretenimiento -> Ocio
                        elif categoria_orig == "Entretenimiento":
                            cat_final = "Ocio"
                    
                    data_list.append({
                        "Fecha": fecha,
                        "Tipo": tipo_movimiento,
                        "Categoría": cat_final,
                        "Descripción": desc_final,
                        "Importe": f"{importe:.2f}".replace('.', ','),
                        "Compromiso": "Variable",
                        "Estado": "Pagado"
                    })

    # Procesar bloques
    extraer_movimientos(idx_ingresos + 1, idx_gastos, "Ingreso")
    extraer_movimientos(idx_gastos + 1, len(df_raw), "Gasto")

    return pd.DataFrame(data_list)

# --- PROCESAR ENERO Y FEBRERO 2023 ---
archivos_2023 = [
    ('../data/2023/2023 - Enero.csv', 2023, 1),
    ('../data/2023/2023 - Febrero.csv', 2023, 2)
]

dfs = []
for path, y, m in archivos_2023:
    if os.path.exists(path):
        print(f"Procesando {path}...")
        dfs.append(procesar_csv_2023(path, y, m))

# Unir todo en un solo archivo
if dfs:
    df_final_2023 = pd.concat(dfs, ignore_index=True)
    
    # Guardar con el formato de comillas y comas que pediste
    df_final_2023.to_csv('2023_limpio.csv', 
                         index=False, 
                         sep=',', 
                         quoting=csv.QUOTE_NONNUMERIC,
                         encoding='utf-8-sig')
    
    print(f"\n¡Hecho! Se han procesado {len(df_final_2023)} movimientos.")
    print(df_final_2023.head())

SyntaxError: invalid syntax (809724837.py, line 110)

In [ ]:

import pandas as pd

import numpy as np

import csv

import os



def procesar_csv_exacto(file_path, year, month):

    df_raw = pd.read_csv(file_path, header=None)

    

    # 1. Encontrar la fila donde están los números de los días (0, 1, 2...)

    idx_dias = None

    for r in range(len(df_raw)):

        fila = df_raw.iloc[r].astype(str).tolist()

        if '0' in fila or '1' in fila:

            idx_dias = r

            break

            

    # 2. Localizar las filas de inicio de Ingresos y Gastos

    # Buscamos en la columna 0 (A)

    try:

        idx_ingresos = df_raw[df_raw[0].astype(str).str.strip() == 'Ingresos'].index[0]

        idx_gastos = df_raw[df_raw[0].astype(str).str.strip() == 'Gastos'].index[0]

    except:

        print(f"⚠️ No se han encontrado las etiquetas en {file_path}")

        return pd.DataFrame()



    data_list = []



    def extraer_datos(start_row, end_row, tipo_mov):

        for i in range(start_row, end_row):

            # La descripción/categoría original está en la columna 1 (B)

            cat_orig = str(df_raw.iloc[i, 1]).strip()

            

            # Saltamos basura y totales

            if cat_orig.lower() in ['nan', '', 'total', 'ahorro', 'diferencia']:

                continue

            

            # Recorrer días (de la columna 2 a la penúltima)

            for col in range(2, df_raw.shape[1] - 1):

                valor = df_raw.iloc[i, col]

                

                if pd.notna(valor) and str(valor).strip() not in ["0,00 €", "0", "0.0", ""]:

                    try:

                        # Limpiar importe: "1,30 €" -> 1.30

                        val_str = str(valor).replace(' €', '').replace('.', '').replace(',', '.')

                        importe = abs(float(val_str))

                        if importe == 0: continue

                    except: continue

                    

                    # Obtener día de la fila de cabecera

                    try:

                        dia = int(float(str(df_raw.iloc[idx_dias, col]).strip()))

                    except: continue

                    

                    fecha = f"{dia:02d}/{month:02d}/{year}"

                    

                    # --- LÓGICA DE CATEGORÍAS EXACTA ---

                    cat_final = cat_orig

                    desc_final = cat_orig

                    

                    if tipo_mov == "Ingreso":

                        cat_final = "Billetazo"

                        # Si es Banca o Ingreso, la descripción suele ser Bizum en tus ejemplos

                        if cat_orig in ["Banca", "Ingreso"]:

                            desc_final = "Bizum"

                    else:

                        # REGLA CAFÉ: Comida entre 1,20 y 1,35 -> Ocio / Café

                        if cat_orig == "Comida" and 1.20 <= importe <= 1.35:

                            cat_final = "Ocio"

                            desc_final = "Café"

                        # Regla Banca -> Otro

                        elif cat_orig == "Banca":

                            cat_final = "Otro"

                        # Regla Entretenimiento -> Ocio

                        elif cat_orig == "Entretenimiento":

                            cat_final = "Ocio"

                    

                    data_list.append({

                        "Fecha": fecha,

                        "Tipo": tipo_mov,

                        "Categoría": cat_final,

                        "Descripción": desc_final,

                        "Importe": f"{importe:.2f}".replace('.', ','),

                        "Compromiso": "Variable",

                        "Estado": "Pagado"

                    })



    # Extraer Ingresos (entre la fila 'Ingresos' y 'Gastos')

    extraer_datos(idx_ingresos, idx_gastos, "Ingreso")

    # Extraer Gastos (desde la fila 'Gastos' hasta el final)

    extraer_datos(idx_gastos, len(df_raw), "Gasto")



    return pd.DataFrame(data_list)



# --- EJECUCIÓN ---

archivo = '../data/2023/2023 - Marzo.csv' 

archivo = '../data/2023/2023 - Abril.csv' 

archivo = '../data/2023/2023 - Junio.csv' 

archivo = '../data/2023/2023 - Julio.csv' 

archivo = '../data/2023/2023 - Agosto.csv' 

archivo = '../data/2023/2023 - Septiembre.csv' 

archivo = '../data/2023/2023 - Octubre.csv' 

archivo = '../data/2023/2023 - Noviembre.csv' 

archivo = '../data/2023/2023 - Diciembre.csv' 

df_final = procesar_csv_exacto(archivo, 2023, 3)



if not df_final.empty:

    df_final.to_csv('mes_2023_limpio.csv', 

                    index=False, 

                    sep=',', 

                    quoting=csv.QUOTE_NONNUMERIC,

                    encoding='utf-8-sig')

    print("✅ Archivo 'mes_2023_limpio.csv' generado con éxito.")

    print(df_final.head())

Procesando: ../data/2023/2023 - Marzo.csv...
Procesando: ../data/2023/2023 - Abril.csv...
Procesando: ../data/2023/2023 - Mayo.csv...
Procesando: ../data/2023/2023 - Junio.csv...
Procesando: ../data/2023/2023 - Julio.csv...
Procesando: ../data/2023/2023 - Agosto.csv...
Procesando: ../data/2023/2023 - Septiembre.csv...
Procesando: ../data/2023/2023 - Octubre.csv...
Procesando: ../data/2023/2023 - Noviembre.csv...
Procesando: ../data/2023/2023 - Diciembre.csv...

✅ ¡Éxito! Se han unido 10 meses.
Total de movimientos: 187
Archivo guardado como: 'historico_2023_completo.csv'


In [1]:
import pandas as pd
import numpy as np
import csv
import os

def procesar_csv_exacto(file_path, year, month):
    if not os.path.exists(file_path):
        print(f"⚠️ Archivo no encontrado: {file_path}")
        return pd.DataFrame()

    df_raw = pd.read_csv(file_path, header=None)
    
    # 1. Encontrar la fila donde están los números de los días (0, 1, 2...)
    idx_dias = None
    for r in range(len(df_raw)):
        fila = df_raw.iloc[r].astype(str).tolist()
        if '0' in fila or '1' in fila:
            idx_dias = r
            break
            
    # 2. Localizar las filas de inicio de Ingresos y Gastos
    try:
        idx_ingresos = df_raw[df_raw[0].astype(str).str.strip() == 'Ingresos'].index[0]
        idx_gastos = df_raw[df_raw[0].astype(str).str.strip() == 'Gastos'].index[0]
    except:
        print(f"⚠️ No se han encontrado las etiquetas 'Ingresos'/'Gastos' en {file_path}")
        return pd.DataFrame()

    data_list = []

    def extraer_datos(start_row, end_row, tipo_mov):
        for i in range(start_row, end_row):
            cat_orig = str(df_raw.iloc[i, 1]).strip()
            
            if cat_orig.lower() in ['nan', '', 'total', 'ahorro', 'diferencia']:
                continue
            
            for col in range(2, df_raw.shape[1] - 1):
                valor = df_raw.iloc[i, col]
                
                if pd.notna(valor) and str(valor).strip() not in ["0,00 €", "0", "0.0", ""]:
                    try:
                        val_str = str(valor).replace(' €', '').replace('.', '').replace(',', '.')
                        importe = abs(float(val_str))
                        if importe == 0: continue
                    except: continue
                    
                    try:
                        dia = int(float(str(df_raw.iloc[idx_dias, col]).strip()))
                    except: continue
                    
                    fecha = f"{dia:02d}/{month:02d}/{year}"
                    
                    cat_final = cat_orig
                    desc_final = cat_orig
                    
                    if tipo_mov == "Ingreso":
                        cat_final = "Billetazo"
                        if cat_orig in ["Banca", "Ingreso"]:
                            desc_final = "Bizum"
                    else:
                        if cat_orig == "Comida" and 1.20 <= importe <= 1.35:
                            cat_final = "Ocio"
                            desc_final = "Café"
                        elif cat_orig == "Banca":
                            cat_final = "Otro"
                        elif cat_orig == "Entretenimiento":
                            cat_final = "Ocio"
                    
                    data_list.append({
                        "Fecha": fecha,
                        "Tipo": tipo_mov,
                        "Categoría": cat_final,
                        "Descripción": desc_final,
                        "Importe": f"{importe:.2f}".replace('.', ','),
                        "Compromiso": "Variable",
                        "Estado": "Pagado"
                    })

    extraer_datos(idx_ingresos, idx_gastos, "Ingreso")
    extraer_datos(idx_gastos, len(df_raw), "Gasto")

    return pd.DataFrame(data_list)

# --- CONFIGURACIÓN DE PROCESAMIENTO POR LOTES ---

# Lista de archivos con su respectivo (Ruta, Año, Mes)
archivos_a_procesar = [
    ('../data/2023/2023 - Marzo.csv', 2023, 3),
    ('../data/2023/2023 - Abril.csv', 2023, 4),
    ('../data/2023/2023 - Mayo.csv', 2023, 5),
    ('../data/2023/2023 - Junio.csv', 2023, 6),
    ('../data/2023/2023 - Julio.csv', 2023, 7),
    ('../data/2023/2023 - Agosto.csv', 2023, 8),
    ('../data/2023/2023 - Septiembre.csv', 2023, 9),
    ('../data/2023/2023 - Octubre.csv', 2023, 10),
    ('../data/2023/2023 - Noviembre.csv', 2023, 11),
    ('../data/2023/2023 - Diciembre.csv', 2023, 12),
]

todos_los_meses = []

for ruta, anio, mes in archivos_a_procesar:
    print(f"Procesando: {ruta}...")
    df_mes = procesar_csv_exacto(ruta, anio, mes)
    if not df_mes.empty:
        todos_los_meses.append(df_mes)

# Concatenar todos los dataframes de la lista en uno solo
if todos_los_meses:
    df_final_unido = pd.concat(todos_los_meses, ignore_index=True)
    
    # Guardar el resultado final
    df_final_unido.to_csv('historico_2023_completo.csv', 
                          index=False, 
                          sep=',', 
                          quoting=csv.QUOTE_NONNUMERIC,
                          encoding='utf-8-sig')
    
    print(f"\n✅ ¡Éxito! Se han unido {len(todos_los_meses)} meses.")
    print(f"📁 Archivo generado: 'historico_2023_completo.csv' con {len(df_final_unido)} registros.")
else:
    print("❌ No se pudo procesar ningún archivo.")

Procesando: ../data/2023/2023 - Marzo.csv...
Procesando: ../data/2023/2023 - Abril.csv...
Procesando: ../data/2023/2023 - Mayo.csv...
Procesando: ../data/2023/2023 - Junio.csv...
Procesando: ../data/2023/2023 - Julio.csv...
Procesando: ../data/2023/2023 - Agosto.csv...
Procesando: ../data/2023/2023 - Septiembre.csv...
Procesando: ../data/2023/2023 - Octubre.csv...
Procesando: ../data/2023/2023 - Noviembre.csv...
Procesando: ../data/2023/2023 - Diciembre.csv...

✅ ¡Éxito! Se han unido 10 meses.
📁 Archivo generado: 'historico_2023_completo.csv' con 445 registros.


In [5]:
import pandas as pd
import numpy as np
import csv
import os

def procesar_csv_exacto(file_path, year, month):
    if not os.path.exists(file_path):
        print(f"⚠️ Archivo no encontrado: {file_path}")
        return pd.DataFrame()

    df_raw = pd.read_csv(file_path, header=None)
    
    # 1. Encontrar la fila donde están los números de los días (0, 1, 2...)
    idx_dias = None
    for r in range(len(df_raw)):
        fila = df_raw.iloc[r].astype(str).tolist()
        if '0' in fila or '1' in fila:
            idx_dias = r
            break
            
    # 2. Localizar las filas de inicio de Ingresos y Gastos
    try:
        idx_ingresos = df_raw[df_raw[0].astype(str).str.strip() == 'Ingresos'].index[0]
        idx_gastos = df_raw[df_raw[0].astype(str).str.strip() == 'Gastos'].index[0]
    except:
        print(f"⚠️ No se han encontrado las etiquetas 'Ingresos'/'Gastos' en {file_path}")
        return pd.DataFrame()

    data_list = []

    def extraer_datos(start_row, end_row, tipo_mov):
        for i in range(start_row, end_row):
            cat_orig = str(df_raw.iloc[i, 1]).strip()
            
            if cat_orig.lower() in ['nan', '', 'total', 'ahorro', 'diferencia']:
                continue
            
            for col in range(2, df_raw.shape[1] - 1):
                valor = df_raw.iloc[i, col]
                
                if pd.notna(valor) and str(valor).strip() not in ["0,00 €", "0", "0.0", ""]:
                    try:
                        val_str = str(valor).replace(' €', '').replace('.', '').replace(',', '.')
                        importe = abs(float(val_str))
                        if importe == 0: continue
                    except: continue
                    
                    try:
                        dia = int(float(str(df_raw.iloc[idx_dias, col]).strip()))
                    except: continue
                    
                    fecha = f"{dia:02d}/{month:02d}/{year}"
                    
                    cat_final = cat_orig
                    desc_final = cat_orig
                    
                    if tipo_mov == "Ingreso":
                        cat_final = "Billetazo"
                        if cat_orig in ["Banca", "Ingreso"]:
                            desc_final = "Bizum"
                    else:
                        if cat_orig == "Comida" and 1.20 <= importe <= 1.45:
                            cat_final = "Ocio"
                            desc_final = "Café"
                        elif cat_orig == "Banca":
                            cat_final = "Otro"
                        elif cat_orig == "Entretenimiento":
                            cat_final = "Ocio"
                    
                    data_list.append({
                        "Fecha": fecha,
                        "Tipo": tipo_mov,
                        "Categoría": cat_final,
                        "Descripción": desc_final,
                        "Importe": f"{importe:.2f}".replace('.', ','),
                        "Compromiso": "Variable",
                        "Estado": "Pagado"
                    })

    extraer_datos(idx_ingresos, idx_gastos, "Ingreso")
    extraer_datos(idx_gastos, len(df_raw), "Gasto")

    return pd.DataFrame(data_list)

# --- CONFIGURACIÓN DE PROCESAMIENTO POR LOTES ---

# Lista de archivos con su respectivo (Ruta, Año, Mes)
archivos_a_procesar = [
    ('../data/2024/2024 - Enero.csv', 2024, 1),
    ('../data/2024/2024 - Febrero.csv', 2024, 2),
    ('../data/2024/2024 - Marzo.csv', 2024, 3),
    ('../data/2024/2024 - Abril.csv', 2024, 4),
    ('../data/2024/2024 - Mayo.csv', 2024, 5),
    ('../data/2024/2024 - Junio.csv', 2024, 6),
    ('../data/2024/2024 - Julio.csv', 2024, 7),
    ('../data/2024/2024 - Agosto.csv', 2024, 8),
    ('../data/2024/2024 - Septiembre.csv', 2024, 9),
    ('../data/2024/2024 - Octubre.csv', 2024, 10),
    ('../data/2024/2024 - Noviembre.csv', 2024, 11),
    ('../data/2024/2024 - Diciembre.csv', 2024, 12),
]

todos_los_meses = []

for ruta, anio, mes in archivos_a_procesar:
    print(f"Procesando: {ruta}...")
    df_mes = procesar_csv_exacto(ruta, anio, mes)
    if not df_mes.empty:
        todos_los_meses.append(df_mes)

# Concatenar todos los dataframes de la lista en uno solo
if todos_los_meses:
    df_final_unido = pd.concat(todos_los_meses, ignore_index=True)
    
    # Guardar el resultado final
    df_final_unido.to_csv('historico_2024_completo.csv', 
                          index=False, 
                          sep=',', 
                          quoting=csv.QUOTE_NONNUMERIC,
                          encoding='utf-8-sig')
    
    print(f"\n✅ ¡Éxito! Se han unido {len(todos_los_meses)} meses.")
    print(f"📁 Archivo generado: 'historico_2024_completo.csv' con {len(df_final_unido)} registros.")
else:
    print("❌ No se pudo procesar ningún archivo.")

Procesando: ../data/2024/2024 - Enero.csv...
Procesando: ../data/2024/2024 - Febrero.csv...
Procesando: ../data/2024/2024 - Marzo.csv...
Procesando: ../data/2024/2024 - Abril.csv...
Procesando: ../data/2024/2024 - Mayo.csv...
Procesando: ../data/2024/2024 - Junio.csv...
Procesando: ../data/2024/2024 - Julio.csv...
Procesando: ../data/2024/2024 - Agosto.csv...
Procesando: ../data/2024/2024 - Septiembre.csv...
Procesando: ../data/2024/2024 - Octubre.csv...
Procesando: ../data/2024/2024 - Noviembre.csv...
Procesando: ../data/2024/2024 - Diciembre.csv...

✅ ¡Éxito! Se han unido 12 meses.
📁 Archivo generado: 'historico_2024_completo.csv' con 539 registros.


In [4]:
import pandas as pd
import numpy as np
import csv
import os

def procesar_csv_exacto(file_path, year, month):
    if not os.path.exists(file_path):
        print(f"⚠️ Archivo no encontrado: {file_path}")
        return pd.DataFrame()

    df_raw = pd.read_csv(file_path, header=None)
    
    # 1. Encontrar la fila donde están los números de los días (0, 1, 2...)
    idx_dias = None
    for r in range(len(df_raw)):
        fila = df_raw.iloc[r].astype(str).tolist()
        if '0' in fila or '1' in fila:
            idx_dias = r
            break
            
    # 2. Localizar las filas de inicio de Ingresos y Gastos
    try:
        idx_ingresos = df_raw[df_raw[0].astype(str).str.strip() == 'Ingresos'].index[0]
        idx_gastos = df_raw[df_raw[0].astype(str).str.strip() == 'Gastos'].index[0]
    except:
        print(f"⚠️ No se han encontrado las etiquetas 'Ingresos'/'Gastos' en {file_path}")
        return pd.DataFrame()

    data_list = []

    def extraer_datos(start_row, end_row, tipo_mov):
        for i in range(start_row, end_row):
            cat_orig = str(df_raw.iloc[i, 1]).strip()
            
            if cat_orig.lower() in ['nan', '', 'total', 'ahorro', 'diferencia']:
                continue
            
            for col in range(2, df_raw.shape[1] - 1):
                valor = df_raw.iloc[i, col]
                
                if pd.notna(valor) and str(valor).strip() not in ["0,00 €", "0", "0.0", ""]:
                    try:
                        val_str = str(valor).replace(' €', '').replace('.', '').replace(',', '.')
                        importe = abs(float(val_str))
                        if importe == 0: continue
                    except: continue
                    
                    try:
                        dia = int(float(str(df_raw.iloc[idx_dias, col]).strip()))
                    except: continue
                    
                    fecha = f"{dia:02d}/{month:02d}/{year}"
                    
                    cat_final = cat_orig
                    desc_final = cat_orig
                    
                    if tipo_mov == "Ingreso":
                        cat_final = "Billetazo"
                        if cat_orig in ["Banca", "Ingreso"]:
                            desc_final = "Bizum"
                    else:
                        if cat_orig == "Comida" and 1.40 <= importe <= 1.55:
                            cat_final = "Ocio"
                            desc_final = "Café"
                        elif cat_orig == "Banca":
                            cat_final = "Otro"
                        elif cat_orig == "Entretenimiento":
                            cat_final = "Ocio"
                    
                    data_list.append({
                        "Fecha": fecha,
                        "Tipo": tipo_mov,
                        "Categoría": cat_final,
                        "Descripción": desc_final,
                        "Importe": f"{importe:.2f}".replace('.', ','),
                        "Compromiso": "Variable",
                        "Estado": "Pagado"
                    })

    extraer_datos(idx_ingresos, idx_gastos, "Ingreso")
    extraer_datos(idx_gastos, len(df_raw), "Gasto")

    return pd.DataFrame(data_list)

# --- CONFIGURACIÓN DE PROCESAMIENTO POR LOTES ---

# Lista de archivos con su respectivo (Ruta, Año, Mes)
archivos_a_procesar = [
    ('../data/2025/2025 - Enero.csv', 2025, 1),
    ('../data/2025/2025 - Febrero.csv', 2025, 2),
    ('../data/2025/2025 - Marzo.csv', 2025, 3),
    ('../data/2025/2025 - Abril.csv', 2025, 4),
    ('../data/2025/2025 - Mayo.csv', 2025, 5),
    ('../data/2025/2025 - Junio.csv', 2025, 6),
    ('../data/2025/2025 - Julio.csv', 2025, 7),
    ('../data/2025/2025 - Agosto.csv', 2025, 8),
    ('../data/2025/2025 - Septiembre.csv', 2025, 9),
    ('../data/2025/2025 - Octubre.csv', 2025, 10),
    ('../data/2025/2025 - Noviembre.csv', 2025, 11),
    ('../data/2025/2025 - Diciembre.csv', 2025, 12),
]

todos_los_meses = []

for ruta, anio, mes in archivos_a_procesar:
    print(f"Procesando: {ruta}...")
    df_mes = procesar_csv_exacto(ruta, anio, mes)
    if not df_mes.empty:
        todos_los_meses.append(df_mes)

# Concatenar todos los dataframes de la lista en uno solo
if todos_los_meses:
    df_final_unido = pd.concat(todos_los_meses, ignore_index=True)
    
    # Guardar el resultado final
    df_final_unido.to_csv('historico_2025_completo.csv', 
                          index=False, 
                          sep=',', 
                          quoting=csv.QUOTE_NONNUMERIC,
                          encoding='utf-8-sig')
    
    print(f"\n✅ ¡Éxito! Se han unido {len(todos_los_meses)} meses.")
    print(f"📁 Archivo generado: 'historico_2025_completo.csv' con {len(df_final_unido)} registros.")
else:
    print("❌ No se pudo procesar ningún archivo.")

Procesando: ../data/2025/2025 - Enero.csv...
Procesando: ../data/2025/2025 - Febrero.csv...
Procesando: ../data/2025/2025 - Marzo.csv...
Procesando: ../data/2025/2025 - Abril.csv...
Procesando: ../data/2025/2025 - Mayo.csv...
Procesando: ../data/2025/2025 - Junio.csv...
Procesando: ../data/2025/2025 - Julio.csv...
Procesando: ../data/2025/2025 - Agosto.csv...
Procesando: ../data/2025/2025 - Septiembre.csv...
Procesando: ../data/2025/2025 - Octubre.csv...
Procesando: ../data/2025/2025 - Noviembre.csv...
Procesando: ../data/2025/2025 - Diciembre.csv...

✅ ¡Éxito! Se han unido 12 meses.
📁 Archivo generado: 'historico_2025_completo.csv' con 473 registros.
